# SDP ETL - IT Support (Silver e Gold)

**Pipeline Declarativo (SDP)** — abordagem moderna de ETL com Lakeflow Spark Declarative Pipelines.

Este notebook define um pipeline completo **Bronze → Silver → Gold** usando:
- **STREAMING TABLE** para a camada Silver (ingestão incremental + regras de qualidade)
- **MATERIALIZED VIEW** para a camada Gold (agregações auto-gerenciadas)
- **Expectations** para validação de qualidade de dados em cada camada

**Source Schema:** Configurado via `source_schema` no pipeline settings (Configuration)  
**Target Schema:** Configurado nos campos **Target catalog** e **Target schema** do pipeline

### ⚙️ Configuração do Pipeline
Ao criar o pipeline, adicione em **Settings → Configuration** (Advanced):
| Chave | Valor | Descrição |
| --- | --- | --- |
| `source_schema` | `<seu_catalog>.<seu_schema>` | Schema onde estão as tabelas Bronze (tb_*) (ex: `workshop_catalog.it_support`) |

### Modos de Violation das Expectations
| Modo | Comportamento | Usado em |
| --- | --- | --- |
| `ON VIOLATION DROP ROW` | Remove linhas inválidas silenciosamente | Silver (limpeza) |
| `ON VIOLATION FAIL UPDATE` | Falha o pipeline se houver violação | Gold (dados críticos) |
| Sem cláusula (WARN) | Registra métrica mas não bloqueia | Gold (monitoramento) |

In [0]:
%sql

-- ==========================================================
-- CAMADA SILVER: Tickets enriquecidos com atendente e SLA
-- STREAMING TABLE = ingestão incremental (só processa novos)
-- Expectations com DROP ROW = remove linhas inválidas
-- ==========================================================

CREATE OR REFRESH STREAMING TABLE sdp_silver_tickets_full (

  -- Regras de qualidade (Expectations)
  CONSTRAINT valid_ticket_id
    EXPECT (ticket_id IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_country
    EXPECT (country IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_priority
    EXPECT (priority IN ('Critical', 'High', 'Medium', 'Low',
                         'critical', 'high', 'medium', 'low'))
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_status
    EXPECT (status IN ('Open', 'Closed', 'open', 'closed'))
    ON VIOLATION DROP ROW,

  CONSTRAINT positive_resolution_hours
    EXPECT (resolution_hours IS NULL OR resolution_hours >= 0)
    ON VIOLATION DROP ROW
)
COMMENT 'Silver: tickets com join de atendente + SLA, campos calculados e qualidade validada'
AS
SELECT
    t.ticket_id,
    INITCAP(t.status)   AS status,
    INITCAP(t.priority) AS priority,
    INITCAP(t.source)   AS source,
    t.topic,
    t.created_time,
    t.close_time,
    t.product_group,
    t.support_level,
    t.country,
    t.latitude,
    t.longitude,

    -- Dados do atendente
    a.agent_name,
    a.agent_group,
    a.agent_interactions,

    -- Dados de SLA
    s.expected_sla_to_resolve,
    s.expected_sla_to_first_response,
    s.first_response_time,
    s.sla_for_first_response,
    s.resolution_time,
    s.sla_for_resolution,
    s.survey_results,

    -- Campos calculados
    ROUND(
      (UNIX_TIMESTAMP(t.close_time) - UNIX_TIMESTAMP(t.created_time)) / 3600, 2
    ) AS resolution_hours,

    ROUND(
      (UNIX_TIMESTAMP(s.first_response_time) - UNIX_TIMESTAMP(t.created_time)) / 3600, 2
    ) AS first_response_hours,

    CASE WHEN t.close_time <= s.expected_sla_to_resolve
         THEN TRUE ELSE FALSE END AS sla_resolution_met,

    CASE WHEN s.first_response_time <= s.expected_sla_to_first_response
         THEN TRUE ELSE FALSE END AS sla_first_response_met

FROM STREAM(${source_schema}.tb_tickets) t
LEFT JOIN ${source_schema}.tb_agents a
  ON t.ticket_id = a.ticket_id
LEFT JOIN ${source_schema}.tb_sla s
  ON t.ticket_id = s.ticket_id

In [0]:
%sql

-- ==========================================================
-- CAMADA SILVER: Comentários com contexto do ticket
-- ==========================================================

CREATE OR REFRESH STREAMING TABLE sdp_silver_comments_enriched (

  CONSTRAINT valid_ticket_id
    EXPECT (ticket_id IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT non_empty_comment
    EXPECT (comments IS NOT NULL AND LENGTH(comments) > 0)
    ON VIOLATION DROP ROW,

  CONSTRAINT reasonable_comment_length
    EXPECT (LENGTH(comments) <= 10000)
    -- sem ON VIOLATION = apenas registra métrica (WARN)
)
COMMENT 'Silver: comentários de tickets enriquecidos com contexto (país, prioridade, tópico)'
AS
SELECT
    c.ticket_id,
    c.comments,
    LENGTH(c.comments) AS comment_length,

    -- Contexto do ticket
    INITCAP(t.status)   AS status,
    INITCAP(t.priority) AS priority,
    t.topic,
    t.country,
    t.product_group,
    t.support_level,
    t.created_time

FROM STREAM(${source_schema}.tb_tickets_comments) c
LEFT JOIN ${source_schema}.tb_tickets t
  ON c.ticket_id = t.ticket_id

In [0]:
%sql

-- ==========================================================
-- CAMADA GOLD: KPIs consolidados de suporte
-- MATERIALIZED VIEW = recalculada automaticamente a cada refresh
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW sdp_gold_support_kpis
COMMENT 'Gold: KPIs gerais de suporte — total tickets, MTTR, SLA %, CSAT'
AS
SELECT
    COUNT(ticket_id)                           AS total_tickets,
    COUNT(DISTINCT agent_name)                 AS total_agents,
    ROUND(AVG(resolution_hours), 2)            AS avg_resolution_hours,
    ROUND(AVG(first_response_hours), 2)        AS avg_first_response_hours,
    ROUND(AVG(survey_results), 2)              AS avg_survey_score,

    ROUND(
      SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END)
      / COUNT(ticket_id) * 100, 2
    ) AS sla_resolution_pct,

    ROUND(
      SUM(CASE WHEN sla_first_response_met THEN 1 ELSE 0 END)
      / COUNT(ticket_id) * 100, 2
    ) AS sla_first_response_pct,

    SUM(CASE WHEN status = 'Open'   THEN 1 ELSE 0 END) AS open_tickets,
    SUM(CASE WHEN status = 'Closed' THEN 1 ELSE 0 END) AS closed_tickets,
    ROUND(AVG(agent_interactions), 2)          AS avg_interactions_per_ticket

FROM sdp_silver_tickets_full

In [0]:
%sql

-- ==========================================================
-- CAMADA GOLD: Métricas agregadas por país
-- Expectation WARN = monitora mas não bloqueia
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW sdp_gold_tickets_by_country (

  CONSTRAINT has_tickets
    EXPECT (total_tickets > 0),

  CONSTRAINT valid_sla_range
    EXPECT (sla_compliance_pct BETWEEN 0 AND 100)
)
COMMENT 'Gold: volume, MTTR, CSAT e SLA % por país'
AS
SELECT
    country,
    COUNT(ticket_id)                           AS total_tickets,
    ROUND(AVG(resolution_hours), 2)            AS avg_resolution_hours,
    ROUND(AVG(survey_results), 2)              AS avg_survey_score,

    ROUND(
      SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END)
      / COUNT(ticket_id) * 100, 2
    ) AS sla_compliance_pct,

    COUNT(DISTINCT agent_name)                               AS agents_count,
    SUM(CASE WHEN priority = 'Critical' THEN 1 ELSE 0 END)  AS critical_tickets,
    SUM(CASE WHEN priority = 'High'     THEN 1 ELSE 0 END)  AS high_tickets

FROM sdp_silver_tickets_full
GROUP BY country
ORDER BY total_tickets DESC

In [0]:
%sql

-- ==========================================================
-- CAMADA GOLD: Performance individual por atendente
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW sdp_gold_agent_performance (

  CONSTRAINT valid_agent_name
    EXPECT (agent_name IS NOT NULL)
)
COMMENT 'Gold: performance individual de cada atendente — volume, MTTR, CSAT, SLA %'
AS
SELECT
    agent_name,
    agent_group,
    COUNT(ticket_id)                           AS total_tickets,
    ROUND(AVG(resolution_hours), 2)            AS avg_resolution_hours,
    ROUND(AVG(first_response_hours), 2)        AS avg_first_response_hours,
    ROUND(AVG(survey_results), 2)              AS avg_survey_score,

    ROUND(
      SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END)
      / COUNT(ticket_id) * 100, 2
    ) AS sla_compliance_pct,

    ROUND(AVG(agent_interactions), 2)          AS avg_interactions,
    COUNT(DISTINCT country)                    AS countries_served

FROM sdp_silver_tickets_full
WHERE agent_name IS NOT NULL
GROUP BY agent_name, agent_group
ORDER BY total_tickets DESC

In [0]:
%sql

-- ==========================================================
-- CAMADA GOLD: Conformidade de SLA por prioridade e país
-- Expectation FAIL UPDATE = falha o pipeline se violada
-- ==========================================================

CREATE OR REFRESH MATERIALIZED VIEW sdp_gold_sla_compliance (

  CONSTRAINT valid_sla_pct
    EXPECT (sla_compliance_pct BETWEEN 0 AND 100)
    ON VIOLATION FAIL UPDATE,

  CONSTRAINT valid_ticket_count
    EXPECT (total_tickets > 0)
    ON VIOLATION FAIL UPDATE
)
COMMENT 'Gold: conformidade de SLA por prioridade e país — com validação crítica'
AS
SELECT
    priority,
    country,
    COUNT(ticket_id)  AS total_tickets,

    SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END)  AS tickets_within_sla,
    SUM(CASE WHEN NOT sla_resolution_met THEN 1 ELSE 0 END) AS tickets_breached_sla,

    ROUND(
      SUM(CASE WHEN sla_resolution_met THEN 1 ELSE 0 END)
      / COUNT(ticket_id) * 100, 2
    ) AS sla_compliance_pct,

    ROUND(AVG(resolution_hours), 2)   AS avg_resolution_hours,
    ROUND(AVG(survey_results), 2)     AS avg_survey_score

FROM sdp_silver_tickets_full
GROUP BY priority, country
ORDER BY priority, total_tickets DESC